2026/09/27 01:02:09 [642163] Number of files: 226,076 (reg: 162,412, dir: 63,543, link: 121)

2026/09/27 01:02:09 [642163] Number of created files: 7,246 (reg: 6,815, dir: 393, link: 38)

2026/09/27 01:02:09 [642163] Number of deleted files: 0

2026/09/27 01:02:09 [642163] Number of regular files transferred: 29,741

2026/09/27 01:02:09 [642163] Total file size: 204,937,456,429 bytes

2026/09/27 01:02:09 [642163] Total transferred file size: 1,541,261,548 bytes

2026/09/27 01:02:09 [642163] Literal data: 644,656,632 bytes

2026/09/27 01:02:09 [642163] Matched data: 896,604,916 bytes

2026/09/27 01:02:09 [642163] File list size: 655,275

2026/09/27 01:02:09 [642163] File list generation time: 0.001 seconds

2026/09/27 01:02:09 [642163] File list transfer time: 0.000 seconds

2026/09/27 01:02:09 [642163] Total bytes sent: 208,961,618

2026/09/27 01:02:09 [642163] Total bytes received: 4,317,865

2026/09/27 01:02:09 [642163] sent 208,961,618 bytes  received 4,317,865 bytes  1,659,762.51 bytes/sec

2026/09/27 01:02:09 [642163] total size is 204,937,456,429  speedup is 960.89

Backup completed successfully at Sun 27 Sep 01:02:09 CEST 2026

=== Backup finished at Sun 27 Sep 01:02:09 CEST 2026 ===


In [30]:
import java.time.LocalDateTime
import java.time.ZonedDateTime
import java.time.format.DateTimeFormatter
import java.time.Duration
import java.time.ZoneId
import java.util.Locale

data class FileDistribution(
    val regularFiles: Int,
    val directories: Int,
    val fileLinks: Int,
)

data class Backup(
    val uuid: String = "random UUID",
    val startTime: LocalDateTime? = null,
    val fileName: String,
    val success: Boolean? = null,
    /** Duration in seconds */
    val duration: Int? = null,
    val totalFiles: FileDistribution? = null,
    val createdFiles: FileDistribution? = null,
    val deletedFiles: FileDistribution? = null,
    val transferredRegularFiles: Int? = null,
    val totalFileSize: Long? = null,
    val transferredFileSize: Long? = null,
    val totalBytesSent: Long? = null,
    val totalBytesReceived: Long? = null,
    val bytesPerSecond: Int? = null,
    val speedup: Int? = null,
)

/**
 * @param logLines consists of the first line with the timestamp of the start and then the last 17 lines which contain the statistics after a successful backup
 */
fun parseRsyncLog(fileName: String, logLines: MutableList<String>): Backup {
    val isFailed = logLines.any { it.contains("Backup failed") }
    val validLines = logLines.filter { it.isNotBlank() }

    // Helper to extract dates from either "2026/09/27 01:02:09" OR "Sun 27 Sep 01:02:09 CEST 2026"
    fun extractDate(line: String?): LocalDateTime? {
        if (line == null) return null
        println("Line: $line")
        val formatter = DateTimeFormatter.ofPattern("EEE [ ]d MMM HH:mm:ss zzz yyyy")
        return runCatching {
            ZonedDateTime.parse(line, formatter).toLocalDateTime()
        }
            .onFailure { error -> 
                println("Error: $error")
            }
            .getOrNull()
    }

    // 1. Parse Datetime (Strictly from the first line)
    val parsedStartTime = extractDate(validLines[validLines.size - 2])

    // 2. Parse Duration (Difference between first and last line)
    val endDatetime = extractDate(validLines.lastOrNull())

    val computedDuration = if (parsedStartTime != null && endDatetime != null) {
        val zoneOffset = ZoneId.systemDefault().rules.getOffset(endDatetime)
        (endDatetime.toEpochSecond(zoneOffset) - parsedStartTime.toEpochSecond(zoneOffset)).toInt()
    } else null

    // 3. Early return if backup failed
    if (isFailed) {
        return Backup(
            fileName = fileName,
            startTime = parsedStartTime,
            success = false,
            duration = computedDuration
        )
    }

    // 4. Full parse if backup succeeded
    fun String.toCleanLong(): Long = this.replace(",", "").toLongOrNull() ?: 0L
    fun String.toCleanInt(): Int = this.replace(",", "").substringBefore('.').toIntOrNull() ?: 0

    var totalFiles: FileDistribution? = null
    var createdFiles: FileDistribution? = null
    var deletedFiles: FileDistribution? = null
    var transRegFiles: Int? = null
    var totalSize: Long? = null
    var transSize: Long? = null
    var bytesSent: Long? = null
    var bytesReceived: Long? = null
    var bytesPerSec: Int? = null
    var parsedSpeedup: Int? = null

    fun parseFileDistribution(line: String): FileDistribution {
        if (line.contains("files: 0")) return FileDistribution(0, 0, 0)

        val parts = line.substringAfter("(").substringBefore(")").split(", ")
        var regCount = 0
        var dirCount = 0
        var linkCount = 0
        parts.forEach { part ->
            when {
                part.startsWith("reg:") -> {
                    regCount = part.substringAfter("reg: ").toCleanInt()
                }
                part.startsWith("dir:") -> {
                    dirCount = part.substringAfter("dir: ").toCleanInt()
                }
                part.startsWith("link:") -> {
                    linkCount = part.substringAfter("link: ").toCleanInt()
                }
            }
        }
        return FileDistribution(regCount, dirCount, linkCount)
    }

    validLines.forEach { line ->
        when {
            line.contains("Number of files:") -> totalFiles = parseFileDistribution(line)
            line.contains("Number of created files:") -> createdFiles = parseFileDistribution(line)
            line.contains("Number of deleted files:") -> deletedFiles = parseFileDistribution(line)
            line.contains("Number of regular files transferred:") ->
                transRegFiles = """transferred:\s*([\d,]+)""".toRegex().find(line)?.groupValues?.get(1)?.toCleanInt()
            line.contains("Total file size:") ->
                totalSize = """size:\s*([\d,]+)""".toRegex().find(line)?.groupValues?.get(1)?.toCleanLong()
            line.contains("Total transferred file size:") ->
                transSize = """size:\s*([\d,]+)""".toRegex().find(line)?.groupValues?.get(1)?.toCleanLong()
            line.contains("bytes/sec") -> {
                val match = """sent\s+([\d,]+)\s+bytes\s+received\s+([\d,]+)\s+bytes\s+([\d.,]+)\s+bytes/sec""".toRegex().find(line)
                match?.let {
                    bytesSent = it.groupValues[1].toCleanLong()
                    bytesReceived = it.groupValues[2].toCleanLong()
                    bytesPerSec = it.groupValues[3].toCleanInt()
                }
            }
            line.contains("speedup is") ->
                parsedSpeedup = """speedup is\s+([\d.,]+)""".toRegex().find(line)?.groupValues?.get(1)?.toCleanInt()
        }
    }

    return Backup(
        fileName = fileName,
        startTime = parsedStartTime,
        success = true,
        duration = computedDuration,
        totalFiles = totalFiles,
        createdFiles = createdFiles,
        deletedFiles = deletedFiles,
        transferredRegularFiles = transRegFiles,
        totalFileSize = totalSize,
        transferredFileSize = transSize,
        totalBytesSent = bytesSent,
        totalBytesReceived = bytesReceived,
        bytesPerSecond = bytesPerSec,
        speedup = parsedSpeedup
    )
}

val rawLog = """
    === Starting backup at Sun 27 Sep 01:02:00 CEST 2026 ===
    2026/09/27 01:02:09 [642163] Number of files: 226,076 (reg: 162,412, dir: 63,543, link: 121)
    2026/09/27 01:02:09 [642163] Number of deleted files: 0
    Backup failed at Sun 27 Sep 01:02:15 CEST 2026
""".trimIndent()

val rawLog2 = """
    ssh: connect to host 192.168.178.31 port 22: Connection timed out
    2026/09/20 01:02:15 [231539] rsync: connection unexpectedly closed (0 bytes received so far) [sender]
    rsync: connection unexpectedly closed (0 bytes received so far) [sender]
    2026/09/20 01:02:15 [231539] sent 0 bytes  received 0 bytes  total size 0
    2026/09/20 01:02:15 [231539] rsync error: error in rsync protocol data stream (code 12) at io.c(232) [sender=3.4.1]
    rsync error: error in rsync protocol data stream (code 12) at io.c(232) [sender=3.4.1]
    Backup failed
    Sun  4 Oct 01:00:01 CEST 2026
    Sun 20 Sep 01:02:15 CEST 2026
""".trimIndent()

//val backup = parseRsyncLog("log1", rawLog.split("\n").toMutableList())
val backup2 = parseRsyncLog("log2", rawLog2.split("\n").toMutableList())
backup2

Line: Sun  4 Oct 01:00:01 CEST 2026
Line: Sun 20 Sep 01:02:15 CEST 2026


Backup(uuid=random UUID, startTime=2026-10-04T01:00:01, fileName=log2, success=false, duration=-1209466, totalFiles=null, createdFiles=null, deletedFiles=null, transferredRegularFiles=null, totalFileSize=null, transferredFileSize=null, totalBytesSent=null, totalBytesReceived=null, bytesPerSecond=null, speedup=null)